In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path

from src.nano_maker.skeleton import SkeletonModel
from src.nano_maker.container import configs

from src.nano_maker.modules.nano_printer.smiles_handler import smiles_fingerprint

In [2]:
c = configs.skeleton_config
model = SkeletonModel(n_embd=c['n_embd'], n_head=c['n_head'], n_layers=c['n_layers'],
                      block_size=c['block_size'], map4_res=c['map4_res'], radial_resolution=c['radial_resolution'],
                      l_a=c['l_a'], dropout=c['dropout'])

In [3]:
from src.paths import *
sk_wt_path = PROJECT_ROOT / "src/nano_maker/container/prototype_checkpoint_1.6.pt"

In [4]:
prototype_weights = torch.load(sk_wt_path, map_location="cpu")

In [5]:
print(type(prototype_weights))

<class 'dict'>


In [6]:
model.load_state_dict(prototype_weights["model_state_dict"])

<All keys matched successfully>

In [9]:
device = 'cpu'
block_size = c["block_size"]
radial_resolution = c["radial_resolution"]

In [13]:
sample_smiles = "CC/C(=C(\c1ccc(cc1)O)/c2ccc(cc2)OCCN(C)C)/c3ccccc3"
sample_map4 = torch.tensor(smiles_fingerprint(sample_smiles), dtype=torch.float32).unsqueeze(0)
model.generate(sample_map4)

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_3026648/4155529102.py:1: SyntaxWarning: invalid escape sequence '\c'
  sample_smiles = "CC/C(=C(\c1ccc(cc1)O)/c2ccc(cc2)OCCN(C)C)/c3ccccc3"


[tensor([[50.7969, 53.1791, 53.5695]], grad_fn=<MulBackward0>),
 tensor([[18.3493, 17.5823, 18.2117]], grad_fn=<MulBackward0>),
 tensor([[2.2174, 3.2345, 4.4461]], grad_fn=<MulBackward0>),
 tensor([[ 5.0156, 13.7817, 31.3228]], grad_fn=<MulBackward0>),
 tensor([[ 5.7670, 21.0003, 35.2917]], grad_fn=<MulBackward0>),
 tensor([[ 5.2883, 25.5286, 36.7078]], grad_fn=<MulBackward0>),
 tensor([[ 5.1202, 31.6348, 38.2541]], grad_fn=<MulBackward0>),
 tensor([[ 5.2705, 37.8850, 39.5989]], grad_fn=<MulBackward0>),
 tensor([[ 7.1432, 39.9338, 42.0595]], grad_fn=<MulBackward0>),
 tensor([[ 9.7066, 39.7870, 43.3517]], grad_fn=<MulBackward0>),
 tensor([[11.1937, 41.3162, 43.8937]], grad_fn=<MulBackward0>),
 tensor([[12.2547, 41.8622, 44.9620]], grad_fn=<MulBackward0>),
 tensor([[16.0855, 42.3776, 48.0475]], grad_fn=<MulBackward0>),
 tensor([[21.9342, 41.1652, 49.7672]], grad_fn=<MulBackward0>),
 tensor([[31.1709, 44.7024, 52.5158]], grad_fn=<MulBackward0>),
 tensor([[36.6923, 49.3907, 53.6194]], grad

In [11]:
model.eval()
with torch.no_grad():
    ctx = torch.zeros(1, block_size, 3, device=device)

    smiles_list = [
        ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
        ("Caffeine", "Cn1cnc2c1c(=O)n(c(=O)n2C)C"),
        ("Ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ]

    for name, smi in smiles_list:
        fp = torch.tensor(smiles_fingerprint(smi),
                         dtype=torch.float32).unsqueeze(0).to(device)
        out, _ = model(ctx, fp)
        print(f"{name}: {out.squeeze().tolist()}")

Aspirin: [7.753468036651611, 9.389548301696777, 2.241460084915161]
Caffeine: [3.1862401962280273, 13.572549819946289, 3.41184139251709]
Ibuprofen: [3.942582130432129, 14.684701919555664, 3.8146274089813232]


In [12]:
model.eval()
with torch.no_grad():
    fp = torch.tensor(smiles_fingerprint("CC(=O)Oc1ccccc1C(=O)O"),
                     dtype=torch.float32).unsqueeze(0).to(device)

    # try different context windows - all padding vs partial real context
    ctx_empty = torch.full((1, block_size, 3),
                           float(radial_resolution * 1.5), device=device)
    ctx_mid = torch.full((1, block_size, 3),
                         float(50), device=device)  # mid-sequence
    ctx_late = torch.full((1, block_size, 3),
                          float(10), device=device)  # near end

    for name, ctx in [("empty", ctx_empty), ("mid", ctx_mid), ("late", ctx_late)]:
        out, _ = model(ctx, fp)
        print(f"{name} context: {out.squeeze().tolist()}")

empty context: [50.079193115234375, 49.361610412597656, 51.9390869140625]
mid context: [2.4642770290374756, 2.916513681411743, 2.806009292602539]
late context: [3.435818910598755, 3.2933435440063477, 3.5186991691589355]
